<a href="https://colab.research.google.com/github/abhi01-sys/ML--Journey/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer

In [ ]:
df = sns.load_dataset('titanic')

In [ ]:
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [ ]:
df.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone'],
      dtype='object')

In [ ]:
df.drop(['who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone','fare','parch'],axis=1,inplace=True)

In [ ]:
df

,survived,pclass,sex,age,sibsp,embarked,class
0,0,3,male,22.0,1,S,Third
1,1,1,female,38.0,1,C,First
2,1,3,female,26.0,0,S,Third
3,1,1,female,35.0,1,S,First
4,0,3,male,35.0,0,S,Third
...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,S,Second
887,1,1,female,19.0,0,S,First
888,0,3,female,NaN,1,S,Third
889,1,1,male,26.0,0,C,First


In [ ]:
df.sample(7)

,survived,pclass,sex,age,sibsp,embarked,class
432,1,2,female,42.0,1,S,Second
214,0,3,male,NaN,1,Q,Third
159,0,3,male,NaN,8,S,Third
511,0,3,male,NaN,0,S,Third
49,0,3,female,18.0,1,S,Third
577,1,1,female,39.0,1,S,First
346,1,2,female,40.0,0,S,Second


In [ ]:
df.isnull().sum()

,0
survived,0
pclass,0
sex,0
age,177
sibsp,0
embarked,2
class,0


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
                                                df.drop('survived',axis=1),df['survived'],
                                                test_size=0.2,random_state=42
                                                )

In [ ]:
X_train.head()

,pclass,sex,age,sibsp,embarked,class
331,1,male,45.5,0,S,First
733,2,male,23.0,0,S,Second
382,3,male,32.0,0,S,Third
704,3,male,26.0,1,S,Third
813,3,female,6.0,4,S,Third


In [ ]:
y_train.sample(5)

,survived
613,0
142,1
314,0
178,0
361,0


#Pipeline Making

1.Handling Missing Values

In [ ]:
tnf1 = ColumnTransformer([
                          ('impute_age',SimpleImputer(),[2]),
                          ('impute_embarked',SimpleImputer(strategy='most_frequent'),[4])
                         ],remainder='passthrough').set_output(transform="pandas")

2. One Hot Encoding

In [ ]:
tnf2 = ColumnTransformer([
                          ('ohe_sex_embarked',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[1,4]),
                          ('class',OrdinalEncoder(categories=[['First','Second','Third']]),[5])
                         ],remainder='passthrough').set_output(transform="pandas")

3.Scalling

In [ ]:
tnf3 = ColumnTransformer([
                            ('scale',MinMaxScaler(),slice(0,10))
                         ]).set_output(transform="pandas")

4.Feature Selection

In [ ]:
tnf4 = SelectKBest(score_func=chi2,k=8).set_output(transform="pandas")

5.Train the model

In [ ]:
tnf5 = DecisionTreeClassifier()

AttributeError: 'DecisionTreeClassifier' object has no attribute 'set_output'

#Create Pipeline

In [ ]:
pipe = Pipeline([
                  ('tnf1',tnf1),
                  ('tnf2',tnf2),
                  ('tnf3',tnf3),
                  ('tnf4',tnf4),
                  ('tnf5',tnf5)
                ])

In [ ]:
pipe.fit(X_train,y_train)

Pipeline(steps=[('tnf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [4])])),
                ('tnf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 4]),
                                                 ('class',
                                                  OrdinalEncoder(categories=[['First',
                                                                              'Second',
                                                                              'Third']]),
                                                  [5])])),
                ('tnf3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('tnf4',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x791079e9a5c0>)),
                ('tnf5', DecisionTreeClassifier())])

-> Prediction

In [ ]:
y_pred = pipe.predict(X_test)

In [ ]:
y_pred

array([1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1,
       0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0])

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.6480446927374302

In [ ]:
# cross validation using cross_val_score
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()

np.float64(0.6208115827834138)

In [ ]:
# gridsearchcv
params = {
    'tnf5__max_depth':[1,2,3,4,5,None]
}


In [ ]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tnf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('impute_age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('impute_embarked',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [4])])),
                                       ('tnf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ohe_sex_embarked',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         [1,
                                                                          4]),
                                                                        ('class',
                                                                         OrdinalEncoder(categories=[['First',
                                                                                                     'Second',
                                                                                                     'Third']]),
                                                                         [5])])),
                                       ('tnf3',
                                        ColumnTransformer(transformers=[('scale',
                                                                         MinMaxScaler(),
                                                                         slice(0, 10, None))])),
                                       ('tnf4',
                                        SelectKBest(k=8,
                                                    score_func=<function chi2 at 0x791079e9a5c0>)),
                                       ('tnf5', DecisionTreeClassifier())]),
             param_grid={'tnf5__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

In [ ]:
grid.best_score_

np.float64(0.6208115827834138)

In [ ]:
grid.best_params_

{'tnf5__max_depth': 2}

#Exporting the Pipeline

In [ ]:
# export
import pickle
pickle.dump(pipe,open('pipe.pkl','wb'))